In [1]:
from openai import OpenAI
import base64
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm
client = OpenAI()


In [2]:
csv_path = 'diffusiondb-2m_random_5k-master-annotations_with_binaries.csv'
models = ['gpt-image-1.5']  # state-of-the-art GPT Image model
output_size = '1024x1024'
output_format = 'png'
output_dir_template = '{model}_generated_images'
output_csv_template = '{model}_generated_images.csv'


In [3]:
df = pd.read_csv(csv_path)

# Keep prompt + the specified text_<emotion> columns (explicit order)
emotion_cols = [
    'text_admiration','text_amusement','text_anger','text_annoyance','text_approval','text_caring',
    'text_confusion','text_curiosity','text_desire','text_disappointment','text_disapproval','text_disgust',
    'text_embarrassment','text_excitement','text_fear','text_gratitude','text_grief','text_joy','text_love',
    'text_nervousness','text_optimism','text_pride','text_realization','text_relief','text_remorse','text_sadness',
    'text_surprise'
]

missing = [c for c in emotion_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

selected_cols = ['prompt'] + emotion_cols
df = df[selected_cols].copy()

df.head()


,prompt,text_admiration,text_amusement,text_anger,text_annoyance,text_approval,text_caring,text_confusion,text_curiosity,text_desire,...,text_joy,text_love,text_nervousness,text_optimism,text_pride,text_realization,text_relief,text_remorse,text_sadness,text_surprise
0,"Walter White doing a kickflip over stairs, pho...",0.062840,0.016919,0.000657,0.004400,0.051100,0.000313,0.000777,0.000692,0.000299,...,0.011561,0.000271,0.000271,0.016003,0.000692,0.006134,0.000247,0.000267,0.000400,0.020523
1,beautiful landscape painting by john constable,0.985823,0.000987,0.000293,0.000395,0.057831,0.000261,0.000337,0.000256,0.000256,...,0.009057,0.002601,0.000219,0.000361,0.000942,0.000635,0.000257,0.000219,0.000259,0.002173
2,"a panda wearing metal frame glasses, sweater a...",0.002239,0.001617,0.000972,0.007628,0.014854,0.000434,0.001019,0.000746,0.000286,...,0.003060,0.000379,0.000279,0.000442,0.000336,0.014125,0.000264,0.000258,0.000338,0.001382
3,a hyperrealistic portrait photo of dancing nin...,0.000891,0.005183,0.000660,0.003423,0.009632,0.000328,0.000622,0.000714,0.000285,...,0.005909,0.000284,0.000272,0.001584,0.000335,0.004440,0.000258,0.000267,0.000297,0.002366
4,"the annunciation by Odd Nerdrum, by Francisco ...",0.966003,0.002184,0.000760,0.001295,0.074038,0.000270,0.001963,0.000676,0.000271,...,0.042420,0.011222,0.001422,0.000859,0.000429,0.031704,0.000225,0.000233,0.000355,0.043575


In [4]:
def generate_image(prompt, model, out_path, size='1024x1024'):
    result = client.images.generate(
        model=model,
        prompt=prompt,
        size=size,
    )

    image_base64 = result.data[0].b64_json
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'wb') as f:
        f.write(base64.b64decode(image_base64))


In [11]:
def generate_for_model(df, model):
    out_dir = Path(output_dir_template.format(model=model))
    rows = []

    emotion_cols = [
        'text_admiration','text_amusement','text_anger','text_annoyance','text_approval','text_caring',
        'text_confusion','text_curiosity','text_desire','text_disappointment','text_disapproval','text_disgust',
        'text_embarrassment','text_excitement','text_fear','text_gratitude','text_grief','text_joy','text_love',
        'text_nervousness','text_optimism','text_pride','text_realization','text_relief','text_remorse','text_sadness',
        'text_surprise'
    ]

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'Generating {model}'):
        image_path = out_dir / f'{idx}.{output_format}'

        base_row = {
            'image_path': str(image_path) if image_path.exists() else '',
            'prompt': row['prompt'],
        }
        for c in emotion_cols:
            base_row[c] = row[c]

        if image_path.exists():
            rows.append(base_row)
            continue

        try:
            generate_image(row['prompt'], model, image_path, size=output_size)
            base_row['image_path'] = str(image_path)
        except Exception as e:
            print(f'Error for idx {idx}: {e}')

        rows.append(base_row)

    out_csv = Path(output_csv_template.format(model=model))
    cols = ['image_path', 'prompt'] + emotion_cols
    pd.DataFrame(rows, columns=cols).to_csv(out_csv, index=False)
    return out_csv


In [ ]:
from PIL import Image as PILImage
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torch.nn.functional import softmax

model_checkpoint = "checkpoint-3640"
model = AutoModelForImageClassification.from_pretrained(model_checkpoint)
processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

idx2label = {
    "0": "amusement",
    "1": "awe",
    "2": "contentment",
    "3": "excitement",
    "4": "anger",
    "5": "disgust",
    "6": "fear",
    "7": "sadness"
}

def predict_emotion(image_input):
    """
    Predict the emotion from an image using a fine-tuned Hugging Face model.

    Parameters:
        image_input (str or PIL.Image.Image): The image file path or a PIL image.

    Returns:
        tuple[int, dict[str, float]]: (predicted label index, probability map)
    """
    # Open the image if a file path is provided
    if isinstance(image_input, str):
        image = PILImage.open(image_input)
    else:
        image = image_input

    # Preprocess the image
    inputs = processor(images=image, return_tensors="pt")

    # Perform the prediction
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits, dim=-1)
    probs = probs.squeeze()

    max_idx = torch.argmax(probs, dim=-1).item()
    probs_map = {}

    for class_idx, prob in enumerate(probs):
        probs_map[idx2label[str(class_idx)]] = prob.item()
    # print("Image predicted.")
    return max_idx, probs_map

In [ ]:
from gpt_image_annotator import predict_emotion

img_emotion_cols = [
    'img_amusement', 'img_awe', 'img_contentment', 'img_excitement',
    'img_anger', 'img_disgust', 'img_fear', 'img_sadness'
]

def annotate_csv_images(csv_path):
    df = pd.read_csv(csv_path)
    for col in img_emotion_cols:
        if col not in df.columns:
            df[col] = pd.NA

    base_dir = Path(csv_path).parent
    updated = 0
    skipped = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Annotating images'):
        image_path = row.get('image_path')
        if not isinstance(image_path, str) or not image_path.strip():
            skipped += 1
            continue

        resolved_path = Path(image_path)
        if not resolved_path.is_absolute():
            resolved_path = (base_dir / resolved_path).resolve()
        if not resolved_path.exists():
            skipped += 1
            continue

        try:
            _, probs = predict_emotion(str(resolved_path))
        except Exception as e:
            print(f'Error annotating {resolved_path}: {e}')
            skipped += 1
            continue

        df.at[idx, 'img_amusement'] = probs['amusement']
        df.at[idx, 'img_awe'] = probs['awe']
        df.at[idx, 'img_contentment'] = probs['contentment']
        df.at[idx, 'img_excitement'] = probs['excitement']
        df.at[idx, 'img_anger'] = probs['anger']
        df.at[idx, 'img_disgust'] = probs['disgust']
        df.at[idx, 'img_fear'] = probs['fear']
        df.at[idx, 'img_sadness'] = probs['sadness']
        updated += 1

    df.to_csv(csv_path, index=False)
    print(f'Annotated {updated} images, skipped {skipped}.')
    return csv_path


In [ ]:
for model in models:
    output_csv = generate_for_model(df, model)
    annotated_csv = annotate_csv_images(output_csv)
    print(f'Wrote {annotated_csv}')


Outputs:
- Images: `<model>_generated_images/`
- CSV: `<model>_generated_images.csv` (includes img_* emotion probability columns)
